In [ ]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [ ]:
# 读取文件并提取数据
def extract_reward_data(filename):
    """
    从文件中提取step和reward数据
    """
    steps = []
    rewards = []
    grads = []
    
    # 使用正则表达式匹配需要的行
    pattern = r'\[Step (\d+)/\d+\] reward=([0-9.]+), grad=([0-9.]+)'
    
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            match = re.search(pattern, line)
            if match:
                step = int(match.group(1))
                reward = float(match.group(2))
                grad = float(match.group(3))
                
                steps.append(step)
                rewards.append(reward)
                grads.append(grad)
    
    return steps, rewards, grads

def plot_reward_curve(steps, rewards, grads=None):
    """
    绘制reward随step变化的曲线
    """
    # 创建图形，设置中文字体
    plt.rcParams['font.sans-serif'] = ['SimHei']  # 用来正常显示中文标签
    plt.rcParams['axes.unicode_minus'] = False   # 用来正常显示负号
    
    if grads:
        # 如果有grad数据，创建两个子图
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8))
        
        # 绘制reward曲线
        ax1.plot(steps, rewards, 'b-o', markersize=4, linewidth=1.5, label='Reward')
        ax1.set_xlabel('Step')
        ax1.set_ylabel('Reward')
        ax1.set_title('Reward with step')
        ax1.grid(True, alpha=0.3)
        ax1.legend()
        
        # 绘制grad曲线
        ax2.plot(steps, grads, 'r-s', markersize=4, linewidth=1.5, label='Grad')
        ax2.set_xlabel('Step')
        ax2.set_ylabel('Grad')
        ax2.set_title('Grad with step')
        ax2.grid(True, alpha=0.3)
        ax2.legend()
        
    else:
        # 只绘制reward曲线
        plt.figure(figsize=(10, 6))
        plt.plot(steps, rewards, 'b-o', markersize=4, linewidth=1.5, label='Reward')
        plt.xlabel('Step')
        plt.ylabel('Reward')
        plt.title('Reward with step')
        plt.grid(True, alpha=0.3)
        plt.legend()
    
    plt.tight_layout()
    plt.show()

In [ ]:
# 指定输入文件名（请修改为你的实际文件名）
input_file = 'logs/checkpoints_math500_num_generation8_t0.3.log'  # 请替换为实际文件名


# 提取数据
steps, rewards, grads = extract_reward_data(input_file)

# 打印统计信息
print(f"找到 {len(steps)} 条数据记录")
print(f"Step 范围: {min(steps)} - {max(steps)}")
print(f"Reward 范围: {min(rewards):.4f} - {max(rewards):.4f}")
print(f"平均 Reward: {np.mean(rewards):.4f}")
print(f"Reward 标准差: {np.std(rewards):.4f}")

if grads:
    print(f"\nGrad 范围: {min(grads):.4f} - {max(grads):.4f}")
    print(f"平均 Grad: {np.mean(grads):.4f}")

# 绘制曲线
plot_reward_curve(steps, rewards, grads)

In [ ]:
import re
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

def extract_diversity_data(filename):
    """
    从文件中提取diversity数据
    """
    distinct_answer_num = []
    best_answer_ratio = []
    
    pattern = r'diversity\|\s*distinct_answer_num:\s*(\d+).*best_answer_ratio:\s*([0-9.]+)'
    
    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            match = re.search(pattern, line)
            if match:
                distinct_answer_num.append(int(match.group(1)))
                best_answer_ratio.append(float(match.group(2)))
    
    # 使用数据点索引作为步数 (0, 1, 2, ...)
    steps = list(range(len(distinct_answer_num)))
    
    return steps, distinct_answer_num, best_answer_ratio

def moving_average(data, window_size=5):
    """计算滑动平均"""
    if len(data) < window_size:
        return data
    return np.convolve(data, np.ones(window_size)/window_size, mode='valid')

def plot_with_moving_average(steps, distinct_answer_num, best_answer_ratio, window_size=5):
    """
    绘制两个图表，都使用滑动平均
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))
    
    # 计算滑动平均
    ma_distinct = moving_average(distinct_answer_num, window_size)
    ma_best = moving_average(best_answer_ratio, window_size)
    ma_steps = steps[window_size-1:]
    
    # 图1: distinct_answer_num with moving average
    ax1.plot(ma_steps, ma_distinct, 'b-', linewidth=2, marker='o', markersize=3, 
             label=f'Distinct Answer Num (MA-{window_size})')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Distinct Answer Num')
    ax1.set_title(f'Distinct Answer Number - Moving Average (window={window_size})')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # 图2: best_answer_ratio with moving average
    ax2.plot(ma_steps, ma_best, 'r-', linewidth=2, marker='s', markersize=3,
             label=f'Best Answer Ratio (MA-{window_size})')
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Best Answer Ratio')
    ax2.set_title(f'Best Answer Ratio - Moving Average (window={window_size})')
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    plt.show()


# 指定输入文件名
input_file = '/Users/mc03002/Documents/JustGRPO/logs/checkpoints_math500_num_generation16_t0.5_lr1e-6.log'  # 请替换为实际文件名


# 提取数据
steps, distinct_answer_num, best_answer_ratio = extract_diversity_data(input_file)

# 设置滑动窗口大小
window_size = 16  # 可以修改这个值来调整平滑程度

# 绘制带滑动平均的图表
plot_with_moving_average(steps, distinct_answer_num, best_answer_ratio, window_size)


对比不同解码方式的rollout

In [25]:
import re
import numpy as np
from pathlib import Path
from collections import Counter

def parse_diversity_line(line):
    """
    解析单行diversity数据，提取完整信息
    示例: diversity| distinct_answer_num: 6 | all_answer_num: 8 | distinct_answer_ratio: 0.75 | best_answer_ratio: 0.38 | correct_answer_number: 1 | best_is_correct: 0 | extracted_answers: ['0', '33.33', '22', '33.33', '33.33', '36.25', '24', '42'] | majority_answer: 33.33 | ground_truth_answer: 22
    """
    # 解析基础指标
    pattern = r'diversity\|\s*distinct_answer_num:\s*(\d+).*?all_answer_num:\s*(\d+).*?best_answer_ratio:\s*([0-9.]+).*?correct_answer_number:\s*(\d+).*?best_is_correct:\s*(\d+)'
    match = re.search(pattern, line)
    
    if not match:
        return None
    
    # 提取答案列表
    answers_pattern = r'extracted_answers:\s*\[(.*?)\]'
    answers_match = re.search(answers_pattern, line)
    extracted_answers = []
    if answers_match:
        # 解析答案列表字符串
        answers_str = answers_match.group(1)
        # 处理引号内的内容
        extracted_answers = re.findall(r"'([^']*)'", answers_str)
    
    # 提取多数投票答案
    majority_pattern = r'majority_answer:\s*([^\s|]+)'
    majority_match = re.search(majority_pattern, line)
    majority_answer = majority_match.group(1) if majority_match else None
    
    # 提取真实答案
    truth_pattern = r'ground_truth_answer:\s*([^\s|]+)'
    truth_match = re.search(truth_pattern, line)
    ground_truth = truth_match.group(1) if truth_match else None
    
    return {
        'distinct_answer_num': int(match.group(1)),
        'all_answer_num': int(match.group(2)),
        'best_answer_ratio': float(match.group(3)),
        'correct_answer_number': int(match.group(4)),
        'best_is_correct': int(match.group(5)),
        'extracted_answers': extracted_answers,
        'majority_answer': majority_answer,
        'ground_truth': ground_truth
    }

def calculate_reward_metrics(answers, majority_answer, ground_truth):
    """
    计算reward accuracy和相关指标
    
    Args:
        answers: 所有rollout的答案列表
        majority_answer: 多数投票答案
        ground_truth: 真实答案
    
    Returns:
        dict: 包含各项指标
    """
    n = len(answers)
    if n == 0 or majority_answer is None or ground_truth is None:
        return {
            'reward_accuracy': 0,
            'true_positive_ratio': 0,
            'true_negative_ratio': 0,
            'false_positive_ratio': 0,
            'false_negative_ratio': 0
        }
    
    # 为每个rollout计算估计奖励和真实奖励
    tp = fp = tn = fn = 0
    
    for ans in answers:
        est_reward = 1 if ans == majority_answer else 0  # 估计奖励：是否匹配多数投票
        true_reward = 1 if ans == ground_truth else 0     # 真实奖励：是否真实正确
        
        if est_reward == 1 and true_reward == 1:
            tp += 1  # True Positive: 估计正确且真实正确
        elif est_reward == 1 and true_reward == 0:
            fp += 1  # False Positive: 估计正确但真实错误
        elif est_reward == 0 and true_reward == 1:
            fn += 1  # False Negative: 估计错误但真实正确
        elif est_reward == 0 and true_reward == 0:
            tn += 1  # True Negative: 估计错误且真实错误
    
    # 计算各项指标
    total = tp + fp + tn + fn
    
    # Reward Accuracy: 估计奖励与真实奖励一致的比例
    reward_accuracy = (tp + tn) / total if total > 0 else 0
    
    # True Positive Ratio: TP / (TP + FN)
    true_positive_ratio = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # True Negative Ratio: TN / (TN + FP)
    true_negative_ratio = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    # False Positive Ratio: FP / (FP + TN)
    false_positive_ratio = fp / (fp + tn) if (fp + tn) > 0 else 0
    
    # False Negative Ratio: FN / (FN + TP)
    false_negative_ratio = fn / (fn + tp) if (fn + tp) > 0 else 0
    
    return {
        'reward_accuracy': reward_accuracy,
        'true_positive_ratio': true_positive_ratio,
        'true_negative_ratio': true_negative_ratio,
        'false_positive_ratio': false_positive_ratio,
        'false_negative_ratio': false_negative_ratio,
        # 也返回原始计数以便调试
        'tp_count': tp,
        'fp_count': fp,
        'tn_count': tn,
        'fn_count': fn
    }

def calculate_metrics(data_list):
    """
    从数据列表中计算各项指标
    过滤掉所有rollout相同的样本（无法产生梯度）
    """
    if not data_list:
        return {
            'avg_distinct_num': 0,
            'avg_best_ratio': 0,
            'avg_correct_num': 0,
            'avg_accuracy': 0,
            'voting_accuracy': 0,
            'consistency': 0,
            'num_samples': 0,
            'good_reward_ratio': 0,
            'zero_reward_ratio': 0,
            'bad_reward_ratio': 0,
            'effective_sample_ratio': 0,  # 有效样本比例（distinct_num > 1）
            'effective_reward_accuracy': 0,  # 有效样本中的reward accuracy
            'effective_tp_ratio': 0,
            'effective_fp_ratio': 0,
            'effective_tn_ratio': 0,
            'effective_fn_ratio': 0,
            'overall_reward_accuracy': 0,  # 所有样本（包括无效）的reward accuracy（供参考）
        }
    
    n_samples = len(data_list)
    all_answer_num = data_list[0]['all_answer_num']
    
    # 原有的指标计算
    avg_distinct_num = np.mean([d['distinct_answer_num'] for d in data_list])
    avg_best_ratio = np.mean([d['best_answer_ratio'] for d in data_list])
    avg_correct_num = np.mean([d['correct_answer_number'] for d in data_list])
    avg_accuracy = avg_correct_num / all_answer_num
    voting_accuracy = np.mean([d['best_is_correct'] for d in data_list])
    
    # 样本类型比例
    good_reward_count = zero_reward_count = bad_reward_count = 0
    for d in data_list:
        if d['distinct_answer_num'] == 1:
            zero_reward_count += 1
        elif d['best_is_correct'] == 1:
            good_reward_count += 1
        else:
            bad_reward_count += 1
    
    # ========== 有效样本过滤 ==========
    # 有效样本：distinct_answer_num > 1（有多样性，能产生梯度）
    effective_samples = [d for d in data_list if d['distinct_answer_num'] > 1]
    effective_sample_ratio = len(effective_samples) / n_samples if n_samples > 0 else 0
    
    # 在所有样本上计算overall reward accuracy（包括无效样本）
    total_tp_all = total_fp_all = total_tn_all = total_fn_all = 0
    
    # 在有效样本上计算effective reward metrics
    total_tp_eff = total_fp_eff = total_tn_eff = total_fn_eff = 0
    
    for d in data_list:
        if 'extracted_answers' not in d or not d['majority_answer'] or not d['ground_truth']:
            continue
            
        answers = d['extracted_answers']
        majority = d['majority_answer']
        truth = d['ground_truth']
        is_effective = d['distinct_answer_num'] > 1
        
        for ans in answers:
            est_reward = 1 if ans == majority else 0
            true_reward = 1 if ans == truth else 0
            
            # 统计所有样本
            if est_reward == 1 and true_reward == 1:
                total_tp_all += 1
            elif est_reward == 1 and true_reward == 0:
                total_fp_all += 1
            elif est_reward == 0 and true_reward == 0:
                total_tn_all += 1
            elif est_reward == 0 and true_reward == 1:
                total_fn_all += 1
            
            # 只统计有效样本
            if is_effective:
                if est_reward == 1 and true_reward == 1:
                    total_tp_eff += 1
                elif est_reward == 1 and true_reward == 0:
                    total_fp_eff += 1
                elif est_reward == 0 and true_reward == 0:
                    total_tn_eff += 1
                elif est_reward == 0 and true_reward == 1:
                    total_fn_eff += 1
    
    # 计算overall指标（所有样本）
    total_rollouts_all = total_tp_all + total_fp_all + total_tn_all + total_fn_all
    if total_rollouts_all > 0:
        overall_reward_accuracy = (total_tp_all + total_tn_all) / total_rollouts_all
        overall_tp_ratio = total_tp_all / total_rollouts_all
        overall_fp_ratio = total_fp_all / total_rollouts_all
        overall_tn_ratio = total_tn_all / total_rollouts_all
        overall_fn_ratio = total_fn_all / total_rollouts_all
    else:
        overall_reward_accuracy = overall_tp_ratio = overall_fp_ratio = overall_tn_ratio = overall_fn_ratio = 0
    
    # 计算effective指标（只有有效样本）
    total_rollouts_eff = total_tp_eff + total_fp_eff + total_tn_eff + total_fn_eff
    if total_rollouts_eff > 0:
        effective_reward_accuracy = (total_tp_eff + total_tn_eff) / total_rollouts_eff
        effective_tp_ratio = total_tp_eff / total_rollouts_eff
        effective_fp_ratio = total_fp_eff / total_rollouts_eff
        effective_tn_ratio = total_tn_eff / total_rollouts_eff
        effective_fn_ratio = total_fn_eff / total_rollouts_eff
    else:
        effective_reward_accuracy = effective_tp_ratio = effective_fp_ratio = effective_tn_ratio = effective_fn_ratio = 0
    
    # 验证有效样本的比例之和为1
    if total_rollouts_eff > 0:
        assert abs(effective_tp_ratio + effective_fp_ratio + effective_tn_ratio + effective_fn_ratio - 1.0) < 1e-10, \
               f"有效样本比例之和应为1，实际为{effective_tp_ratio + effective_fp_ratio + effective_tn_ratio + effective_fn_ratio}"
    
    return {
        # 原有指标
        'avg_distinct_num': avg_distinct_num,
        'avg_best_ratio': avg_best_ratio,
        'avg_correct_num': avg_correct_num,
        'avg_accuracy': avg_accuracy,
        'voting_accuracy': voting_accuracy,
        'consistency': avg_best_ratio,
        'num_samples': n_samples,
        'good_reward_ratio': good_reward_count / n_samples,
        'zero_reward_ratio': zero_reward_count / n_samples,
        'bad_reward_ratio': bad_reward_count / n_samples,
        
        # 新增：有效样本比例
        'effective_sample_ratio': effective_sample_ratio,
        
        # 有效样本上的reward metrics
        'effective_reward_accuracy': effective_reward_accuracy,
        'effective_tp_ratio': effective_tp_ratio,
        'effective_fp_ratio': effective_fp_ratio,
        'effective_tn_ratio': effective_tn_ratio,
        'effective_fn_ratio': effective_fn_ratio,
        
        # 整体指标（供参考）
        'overall_reward_accuracy': overall_reward_accuracy,
    }

def print_metrics_table(metrics_list):
    """
    打印指标表格（包含有效样本指标）
    """
    print("\n" + "="*200)
    header = f"{'File':<60} {'Samples':<6} {'Eff%':<6} {'Distinct':<8} {'Consist':<7} {'RollAcc':<7} {'VoteAcc':<7} "
    header += f"{'GoodR':<6} {'ZeroR':<6} {'BadR':<6} "
    header += f"{'EffTP':<6} {'EffFP':<6} {'EffTN':<6} {'EffFN':<6} {'EffRewAcc':<8} {'AllRewAcc':<8}"
    print(header)
    print("="*200)
    
    for metrics in metrics_list:
        if metrics:
            filename = metrics['filename'][:60] + ".." if len(metrics['filename']) > 40 else metrics['filename']
            print(f"{filename:<60} {metrics['num_samples']:<6} "
                  f"{metrics['effective_sample_ratio']:<6.2f} "
                  f"{metrics['avg_distinct_num']:<8.2f} "
                  f"{metrics['consistency']:<7.3f} "
                  f"{metrics['avg_accuracy']:<7.3f} "
                  f"{metrics['voting_accuracy']:<7.3f} "
                  f"{metrics['good_reward_ratio']:<6.3f} "
                  f"{metrics['zero_reward_ratio']:<6.3f} "
                  f"{metrics['bad_reward_ratio']:<6.3f} "
                  f"{metrics['effective_tp_ratio']:<6.3f} "
                  f"{metrics['effective_fp_ratio']:<6.3f} "
                  f"{metrics['effective_tn_ratio']:<6.3f} "
                  f"{metrics['effective_fn_ratio']:<6.3f} "
                  f"{metrics['effective_reward_accuracy']:<8.3f} "
                  f"{metrics['overall_reward_accuracy']:<8.3f}")
    
    print("="*200)
    print("\n指标说明:")
    print("  Eff%:     有效样本比例 (distinct_num>1, 能产生梯度的样本)")
    print("  Distinct: 平均不同答案数量")
    print("  Consist:  一致性 (最常见答案比例)")
    print("  RollAcc:  rollout平均准确率")
    print("  VoteAcc:  投票准确率")
    print("  GoodR:    好样本比例 (投票正确且多样)")
    print("  ZeroR:    零奖励比例 (无多样性)")
    print("  BadR:     坏样本比例 (投票错误)")
    print("  EffTP:    有效样本中估计正确且真实正确比例")
    print("  EffFP:    有效样本中估计正确但真实错误比例") 
    print("  EffTN:    有效样本中估计错误且真实错误比例")
    print("  EffFN:    有效样本中估计错误但真实正确比例")
    print("  EffRewAcc:有效样本中的Reward Accuracy (TP+TN)")
    print("  AllRewAcc:所有样本中的Reward Accuracy (包括无效样本, 供参考)")
    print(f"\n  验证: EffTP+EffFP+EffTN+EffFN = {metrics['effective_tp_ratio']+metrics['effective_fp_ratio']+metrics['effective_tn_ratio']+metrics['effective_fn_ratio']:.3f} (应为1)")

def process_file(filename, max_line=0):
    """
    处理单个文件，提取diversity数据并计算指标
    """
    data_list = []
    
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            for line in f:
                if 'diversity|' in line:
                    data = parse_diversity_line(line)
                    if data:
                        data_list.append(data)
    except FileNotFoundError:
        print(f"Warning: File {filename} not found")
        return None
    
    if not data_list:
        print(f"Warning: No diversity data found in {filename}")
        return None
    if max_line > 0:
        data_list = data_list[:max_line]
    
    metrics = calculate_metrics(data_list)
    metrics['filename'] = Path(filename).name
    
    return metrics


def plot_comparison_bar_chart(metrics_list):
    """
    绘制对比柱状图，横坐标为rollout_block_size
    """
    # 准备数据 - 假设metrics_list的顺序是 [block1, block32, block256]
    block_sizes = ['1', '32', '256']
    
    # 指标数据
    distinct_nums = [m['avg_distinct_num'] for m in metrics_list if m]
    consistencies = [m['consistency'] for m in metrics_list if m]
    rollout_accs = [m['avg_accuracy'] for m in metrics_list if m]
    voting_accs = [m['voting_accuracy'] for m in metrics_list if m]
    
    # 设置图形
    fig, axes = plt.subplots(2, 2, figsize=(6, 5))
    fig.suptitle('Comparison of Different Metrics Across Rollout Block Sizes', fontsize=16, fontweight='bold')
    
    x = np.arange(len(block_sizes))
    width = 0.6
    
    colors = ['#FF9999', '#66B2FF', '#99FF99', '#FFCC99']
    
    # 1. Average Distinct Answer Number
    ax1 = axes[0, 0]
    bars1 = ax1.bar(x, distinct_nums, width, color=colors[0], edgecolor='black', linewidth=1)
    ax1.set_xlabel('Rollout Block Size', fontsize=12)
    ax1.set_ylabel('Distinct Answer', fontsize=12)
    ax1.set_title('Rollout Diversity', fontsize=14, fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels(block_sizes)
    ax1.grid(True, alpha=0.3, axis='y')
    
    # 添加数值标签
    for bar in bars1:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontweight='bold')
    
    # 2. Consistency (best_answer_ratio)
    ax2 = axes[0, 1]
    bars2 = ax2.bar(x, consistencies, width, color=colors[1], edgecolor='black', linewidth=1)
    ax2.set_xlabel('Rollout Block Size', fontsize=12)
    ax2.set_ylabel('Best Answer Ratio', fontsize=12)
    ax2.set_title('Rollout Consistency', fontsize=14, fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels(block_sizes)
    ax2.set_ylim(0, 1.1)
    ax2.grid(True, alpha=0.3, axis='y')
    
    for bar in bars2:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 3. Rollout Accuracy
    ax3 = axes[1, 0]
    bars3 = ax3.bar(x, rollout_accs, width, color=colors[2], edgecolor='black', linewidth=1)
    ax3.set_xlabel('Rollout Block Size', fontsize=12)
    ax3.set_ylabel('Accuracy', fontsize=12)
    ax3.set_title('All Rollouts Accuracy', fontsize=14, fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels(block_sizes)
    ax3.set_ylim(0, 1.1)
    ax3.grid(True, alpha=0.3, axis='y')
    
    for bar in bars3:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 4. Voting Accuracy
    ax4 = axes[1, 1]
    bars4 = ax4.bar(x, voting_accs, width, color=colors[3], edgecolor='black', linewidth=1)
    ax4.set_xlabel('Rollout Block Size', fontsize=12)
    ax4.set_ylabel('Accuracy', fontsize=12)
    ax4.set_title('Voting Results Accuracy', fontsize=14, fontweight='bold')
    ax4.set_xticks(x)
    ax4.set_xticklabels(block_sizes)
    ax4.set_ylim(0, 1.1)
    ax4.grid(True, alpha=0.3, axis='y')
    
    for bar in bars4:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.3f}', ha='center', va='bottom', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def plot_grouped_bar_chart(metrics_list):
    """
    绘制分组柱状图，把四个指标放在一起对比，横坐标为rollout_block_size
    """
    block_sizes = ['1', '32', '256']
    
    # 指标数据
    distinct_nums = [m['avg_distinct_num'] for m in metrics_list if m]
    consistencies = [m['consistency'] for m in metrics_list if m]
    rollout_accs = [m['avg_accuracy'] for m in metrics_list if m]
    voting_accs = [m['voting_accuracy'] for m in metrics_list if m]
    
    # 归一化distinct_num到0-1范围便于比较
    max_distinct = max(distinct_nums) if distinct_nums else 1
    distinct_nums_norm = [d/max_distinct for d in distinct_nums]
    
    # 设置图形
    fig, ax = plt.subplots(figsize=(14, 8))
    
    x = np.arange(len(block_sizes))
    width = 0.2
    
    # 绘制四个指标的柱状图
    bars1 = ax.bar(x - 1.5*width, distinct_nums_norm, width, label='Diversity (normalized)', color='#FF9999', edgecolor='black')
    bars2 = ax.bar(x - 0.5*width, consistencies, width, label='Consistency', color='#66B2FF', edgecolor='black')
    bars3 = ax.bar(x + 0.5*width, rollout_accs, width, label='Rollout Accuracy', color='#99FF99', edgecolor='black')
    bars4 = ax.bar(x + 1.5*width, voting_accs, width, label='Voting Accuracy', color='#FFCC99', edgecolor='black')
    
    ax.set_xlabel('Rollout Block Size', fontsize=14)
    ax.set_ylabel('Value', fontsize=14)
    ax.set_title('Comparison of All Metrics Across Rollout Block Sizes', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(block_sizes)
    ax.set_ylim(0, 1.1)
    ax.legend(loc='upper right', fontsize=12)
    ax.grid(True, alpha=0.3, axis='y')
    
    # 添加数值标签
    for bars in [bars1, bars2, bars3, bars4]:
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.show()


In [27]:

# 指定三个日志文件
# log_files = [
#         'logs/checkpoints_gsm8k_num_generation8_test.log',           # block size 1 (没有block后缀的)
#         'logs/checkpoints_gsm8k_num_generation8_test_block32.log',   # block size 32
#         'logs/checkpoints_gsm8k_num_generation8_test_block256.log'   # block size 256
# ]

log_files = [
    'logs/checkpoints_math500_num_generation8_block1_t0.6_lr1e-6_v2.log'
]

# 处理每个文件
metrics_list = []
for log_file in log_files:
    print(f"Processing {log_file}...")
    metrics = process_file(log_file, max_line=80)
    if metrics:
        metrics_list.append(metrics)


# 打印表格
print_metrics_table(metrics_list)

# # 绘制对比柱状图
# plot_comparison_bar_chart(metrics_list)

# 也可以绘制分组柱状图（可选）
# plot_grouped_bar_chart(metrics_list)



Processing logs/checkpoints_math500_num_generation8_block1_t0.6_lr1e-6_v2.log...

File                                                         Samples Eff%   Distinct Consist RollAcc VoteAcc GoodR  ZeroR  BadR   EffTP  EffFP  EffTN  EffFN  EffRewAcc AllRewAcc
checkpoints_math500_num_generation8_block1_t0.6_lr1e-6_v2.lo.. 80     0.88   3.92     0.593   0.369   0.450   0.350  0.125  0.525  0.256  0.206  0.504  0.035  0.759    0.764   

指标说明:
  Eff%:     有效样本比例 (distinct_num>1, 能产生梯度的样本)
  Distinct: 平均不同答案数量
  Consist:  一致性 (最常见答案比例)
  RollAcc:  rollout平均准确率
  VoteAcc:  投票准确率
  GoodR:    好样本比例 (投票正确且多样)
  ZeroR:    零奖励比例 (无多样性)
  BadR:     坏样本比例 (投票错误)
  EffTP:    有效样本中估计正确且真实正确比例
  EffFP:    有效样本中估计正确但真实错误比例
  EffTN:    有效样本中估计错误且真实错误比例
  EffFN:    有效样本中估计错误但真实正确比例
  EffRewAcc:有效样本中的Reward Accuracy (TP+TN)
  AllRewAcc:所有样本中的Reward Accuracy (包括无效样本, 供参考)

  验证: EffTP+EffFP+EffTN+EffFN = 1.000 (应为1)
